# Decennial PL 94-171 data

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

<div style="text-align: center;"><a class="sd-sphinx-override sd-btn sd-text-wrap sd-btn-primary reference external" href="https://www.dropbox.com/scl/fo/s22x9phl0hldiakn8nbuz/ABKfxHBaak5ra3eBGkNFWMM?rlkey=igpo7qi07oz5tfgjki317o79t&amp;st=gcxkicnc&amp;dl=1">Download tutorial data</a></div>

Every ten years the Census Bureau releases the PL 94-171 redistricting file: the official
population counts states use to draw districts. The `census()` function is the main interface used
to fetch that data.

The live Census API calls on this page are shown as copyable blocks, and small committed extracts
of their responses (in the tutorial `data/census/` directory) stand in for the network so the
notebook runs without credentials. See the [overview](overview.ipynb) for API-key handling.

In [ ]:
from pathlib import Path

import pandas as pd

from gerrytools.data import pl_table

census_dir = Path("data/census")
pd.set_option("display.max_columns", None)

## Total population for a state

`census(state, geometry, year, table)` defaults to geometry `"state"`, year `2020`, and table
`"P1"` (total population by race):

```python
import us

from gerrytools.data import census

georgia_total = census(us.states.GA, table="P1")
```

Table P1 breaks the population into every combination of races, so it is wide (71 columns). Below
is an example of how to extract just the total population, single race, and two-or-more-races 
columns:

In [ ]:
# GA statewide population totals.
georgia_total = pd.read_csv(
    census_dir / "ga_state_P1_2020.csv", dtype={"GEOID": "string"}, index_col="GEOID"
)

single_race = [
    "total_pop_20",
    "white_pop_20",
    "black_pop_20",
    "amin_pop_20",
    "asian_pop_20",
    "nhpi_pop_20",
    "other_pop_20",
    "two_or_more_races_pop_20",
]

# Transpose the dataframe to make presentation nicer
georgia_total[single_race].T

The index is the GEOID (`13` is Georgia's state FIPS), and every column is a numeric count. The
`_20` suffix records the vintage, so columns stay unambiguous when 2010 and 2020 pulls meet in one
frame.

## Geographies

Pass `geometry` to change the unit: `"state"`, `"county"`, `"tract"`, or `"block group"`. The
GEOID lengthens as the geography gets finer.

| `geometry` | GEOID width | Example |
| --- | --- | --- |
| `"state"` | 2 | `13` |
| `"county"` | 5 | `13121` |
| `"tract"` | 11 | `13121011500` |
| `"block group"` | 12 | `131210115001` |

Here are Georgia's 159 counties:

```python
ga_counties = census(us.states.GA, geometry="county", table="P1")
```

In [ ]:
ga_counties = pd.read_csv(
    census_dir / "ga_county_P1_2020.csv", dtype={"GEOID": "string"}, index_col="GEOID"
)
print("shape:", ga_counties.shape)
ga_counties[["total_pop_20", "white_pop_20", "black_pop_20"]].head()

Finer geographies are a one-word change: `geometry="tract"` returns roughly two thousand rows for
Georgia and `geometry="block group"` several thousand, with the same columns and a longer GEOID.

## The PL tables

The PL file ships four population tables plus housing and group-quarters counts, and `census()`
takes any of them as the `table` argument:

| Table | Contents | Example columns |
| --- | --- | --- |
| `"P1"` | Population by race | `total_pop_20`, `black_pop_20` |
| `"P2"` | Population by race and Hispanic origin | `hispanic_pop_20`, `non_hispanic_white_pop_20` |
| `"P3"` | Voting-age population (VAP) by race | `total_vap_20`, `black_vap_20` |
| `"P4"` | VAP by race and Hispanic origin | `hispanic_vap_20`, `non_hispanic_white_vap_20` |
| `"P5"` | Group-quarters population (2020 only) | `adult_correctional_facility_pop_20` |
| `"H1"` | Housing units | `total_housing_units_20`, `occupied_housing_units_20` |

P3 is the redistricting workhorse: voting-age population is the denominator most district-level
demographic shares use.

```python
vap = census(us.states.GA, geometry="county", table="P3")
```

In [ ]:
vap = pd.read_csv(
    census_dir / "ga_county_P3_2020.csv", dtype={"GEOID": "string"}, index_col="GEOID"
)
vap[["total_vap_20", "white_vap_20", "black_vap_20", "asian_vap_20"]].head()

P2 and P4 add the Hispanic-origin split. P2 in particular is wide (every race combination crossed
with Hispanic origin), so select the columns you need.

```python
hispanic = census(us.states.GA, geometry="county", table="P2")
```

In [ ]:
hispanic = pd.read_csv(
    census_dir / "ga_county_P2_2020.csv", dtype={"GEOID": "string"}, index_col="GEOID"
)
hispanic[
    ["total_pop_20", "hispanic_pop_20", "non_hispanic_white_pop_20", "non_hispanic_black_pop_20"]
].head()

## Table-definition objects

The string shortcuts are resolved through `pl_table(table, year)`, which returns a `PLTableInfo`.
Build one yourself to inspect the raw Census variables a table maps, or pass it straight to
`census(table=...)` in place of the string.

In [ ]:
p3_table = pl_table("P3", 2020)
print("table_name:", p3_table.table_name)
rename_map = p3_table.construct_rename_map(year=2020)
print("a few raw Census variables and their names:")
list(rename_map.items())[:5]

`construct_short_names()` lists the semantic names alone, and `census_column_name(name, year=...)`
builds the vintage-suffixed public column name from any semantic base name.

## Vintages: 2010 and 2020

`year` selects the decennial vintage; PL data exists for `2010` and `2020`. The column suffix
tracks the year, so the same code path handles both:

```python
pop_2010 = census(us.states.GA, year=2010, table="P1")
pop_2020 = census(us.states.GA, year=2020, table="P1")
```

The raw variable names differ between vintages (2010's `P001001` vs 2020's `P1_001N`), so each
`PLTableInfo` is pinned to its year: passing a table built for one vintage with a different
`year` raises a `ValueError` instead of returning an empty frame.

In [ ]:
pop_2010 = pd.read_csv(
    census_dir / "ga_state_P1_2010.csv", dtype={"GEOID": "string"}, index_col="GEOID"
)
pop_2020 = georgia_total

print(f"2010 total: {int(pop_2010['total_pop_10'].iloc[0]):>12,}")
print(f"2020 total: {int(pop_2020['total_pop_20'].iloc[0]):>12,}")

## From table to quantity

Because every result is a GEOID-indexed numeric frame, a real quantity is one division away. Here
is each county's Black share of voting-age population, the kind of number a
[choropleth](../plotting/geographic/geo.ipynb) would map or a [box plot](../plotting/statistical/box.ipynb) would summarize
across an ensemble.

In [ ]:
bvap_share = (vap["black_vap_20"] / vap["total_vap_20"]).rename("bvap_share")
summary = vap[["total_vap_20", "black_vap_20"]].join(bvap_share)
summary.sort_values("bvap_share", ascending=False).head()

## Keys and rate limits

Two errors are worth recognizing:

- **No key.** If neither `api_key=` nor `CENSUS_API_KEY` is set, the call raises a `ValueError`
  telling you to register a key, before any request is made.
- **Rate limit.** If the API returns HTTP 429, `gerrytools` raises `CensusRateLimitError`
  (importable from `gerrytools.data`) with the offending URL and a `Retry-After` hint.

## Related

- [ACS guide](acs.ipynb) for between-census estimates and the citizenship data PL does not
  carry.
- [Block CVAP guide](block_cvap.ipynb) for block-level CVAP built from PL and ACS together.
- [Overview](overview.ipynb) for GEOID handling, merging, and key setup.
- [Data API](../../api/data.rst)